## Rag Implementation in pure python

In [10]:
# Loading the Document
import os, requests

GITHUB_RAW_URL = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt"

def load_document(url: str) -> str:
    """Fetch a plain-text file from a raw GitHub URL."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.text

raw_text = load_document(GITHUB_RAW_URL)
print(f"Loaded {len(raw_text):,} characters")
print(raw_text[:400])  # Sanity check

Loaded 16,864 characters
# AtliqAI HR Policies

AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.

---

## Employment & Onboarding

### Offer and Joi


## Chunking

In [11]:
CHUNK_SIZE = 100

def parse_word_chunks(text: str, chunk_size: int = CHUNK_SIZE) -> list[dict]:
    # Strip markdown heading symbols and blank lines
    clean_lines = []
    for line in text.splitlines():
        line = line.strip().lstrip("#").strip()
        if line:
            clean_lines.append(line)

    # Join everything into one word list and slice
    words = " ".join(clean_lines).split()

    chunks = []
    chunk_index = 0
    for i in range(0, len(words), chunk_size):
        content = " ".join(words[i : i + chunk_size])
        chunks.append({
            "chunk_index": chunk_index,
            "content": content,
        })
        chunk_index += 1

    return chunks

In [12]:
parsed_chunks = parse_word_chunks(raw_text, chunk_size=CHUNK_SIZE)
print(f"Parsed {len(parsed_chunks)} chunks")

Parsed 26 chunks


In [13]:
parsed_chunks

[{'chunk_index': 0,
  'content': 'AtliqAI HR Policies AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining. --- Employment & Onboarding Offer and Joining Formalities Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will share a pre-joining checklist that includes submission of educational certificates, identity proof, address proof, previous employment documents, and a recent'},
 {'chunk_index': 1,
  'content': 'photograph. Failure to submit required documents within 7 working days of joining may result in withholding of the first salary disbursement. Probation Period All new employees at AtliqAI are placed on a probation

In [15]:
# Inspect a chunk
for chunk in parsed_chunks[:3]:
    print("─" * 55)
    print(f"Content : {chunk['content'][:200]}…")
    print(f"Chunk Index : {chunk['chunk_index']}")

───────────────────────────────────────────────────────
Content : AtliqAI HR Policies AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compe…
Chunk Index : 0
───────────────────────────────────────────────────────
Content : photograph. Failure to submit required documents within 7 working days of joining may result in withholding of the first salary disbursement. Probation Period All new employees at AtliqAI are placed o…
Chunk Index : 1
───────────────────────────────────────────────────────
Content : new hires. This includes employment history verification for the last 5 years, educational qualification checks, criminal record screening, and reference checks from at least two previous managers. Th…
Chunk Index : 2


In [10]:
def build_chunk_text(chunk: dict) -> str:
    return chunk["content"]

## Embedding

In [14]:
import tqdm as notebook_tqdm

from sentence_transformers import SentenceTransformer

In [16]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Embedder loaded: {embedder_model.__class__.__name__}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4127.46it/s]


Embedder loaded: SentenceTransformer


In [17]:
# Extract Chunk Texts
chunk_texts = [build_chunk_text(c) for c in parsed_chunks]

print(f"Embedding {len(chunk_texts)} chunks …")
embeddings = embedder_model.encode(chunk_texts, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")

Embedding 26 chunks …


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.40s/it]

Shape: (26, 384)


## Indexing

In [20]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
    )

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "hr_docs"
RESET_COLLECTION = True  # Set False to keep an existing collection and its data.

client = QdrantClient(url=QDRANT_URL, timeout=10)

# Confirm that Qdrant is reachable.
try:
    client.get_collections()
except Exception as exc:
    raise ConnectionError(
        f"Cannot connect to Qdrant at {QDRANT_URL}. "
        "Start Qdrant, then try again."
    ) from exc

DIM = embedder_model.get_embedding_dimension()
if DIM is None:
    raise ValueError("Could not determine the embedding dimension.")

if client.collection_exists(COLLECTION_NAME):
    if RESET_COLLECTION:
        client.delete_collection(COLLECTION_NAME)
        print(f"Deleted existing collection: {COLLECTION_NAME}")
    else:
        print(f"Using existing collection: {COLLECTION_NAME}")

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=DIM,
            distance=Distance.COSINE,
        ),
    )
    print("Collection created.")

Deleted existing collection: hr_docs
Collection created.


In [45]:
x = [i for i in range(1,11)]
y = [i for i in range(11,21)]

In [46]:
for idx, (x, y) in enumerate(zip(x, y)):
    print(idx, (x, y))

0 (1, 11)
1 (2, 12)
2 (3, 13)
3 (4, 14)
4 (5, 15)
5 (6, 16)
6 (7, 17)
7 (8, 18)
8 (9, 19)
9 (10, 20)


In [47]:
# Creating Points

points = [
    PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload={
            "content": chunk["content"],
        },
    )
    for idx, (chunk, embedding) in enumerate(zip(parsed_chunks, embeddings))
]

result = client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,   # Block until indexing completes before returning
)
print(f"Indexed {len(points)} points — status: {result.status}")

Indexed 26 points — status: completed


In [48]:
info = client.get_collection(COLLECTION_NAME)
print(f"Points     : {info.points_count}")
print(f"Dimensions : {info.config.params.vectors.size}")

Points     : 26
Dimensions : 384


## Retrieval

In [49]:
def retrieve(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks.

    Args:
        query          : User's question.
        top_k          : Number of chunks to return.
        section_filter : Optional H2 heading to restrict the search scope.
    """
    query_vector = embedder_model.encode(query).tolist()

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return [{**hit.payload, "score": round(hit.score, 4)} for hit in hits.points]

In [51]:
results = retrieve("What is the leave policy")
for r in results:
    print(f"[score={r['score']}]")
    print(f"  {r['content'][400:]}…\n")

[score=0.4335]
  forward to the following year. Earned Leave Employees accrue earned leave at the rate of 1.25 days per month, amounting to 15 days per year. Earned leave can be carried forward up…

[score=0.4189]
  o 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unplanned absences. A maximum of 3 consecutive casual leaves can be taken at a time. Casual leaves cannot…

[score=0.4009]
  financial year may impact annual performance ratings and bonus eligibility. --- Compensation & Benefits Salary Structure AtliqAI follows a cost-to-company (CTC) model. The salary structure comprises Basic Pay (40% of CTC), House Rent Allowance (20%…

[score=0.4009]
   policy violations — may raise it formally through the HRMS portal under the "Grievance" section. Grievances may also be submitted in writing to the HR Business Partner assigned to the employee's department. Grievance Resolution Timeline HR will acknowle

## RAG Pipeline

In [52]:
SYSTEM_PROMPT = """You are a helpful HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

In [53]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [54]:
import getpass

In [55]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

### End to End RAG Pipeline

In [57]:
from groq import Groq

In [58]:
groq_client = Groq()   # Reads GROQ_API_KEY from environment automatically
GROQ_MODEL  = "openai/gpt-oss-20b"

def rag(query: str, top_k: int = 5):
    """
    End-to-end RAG pipeline:
      1. Retrieve relevant chunks from Qdrant
      2. Format them as a context block
      3. Send context + query to Groq and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    if not chunks:
        return "No relevant content found in the document."

    # Step 2 — Build context
    context = build_context(chunks)

    # Step 3 — Generate
    user_message = f"Context:\n{context}\n\nQuestion: {query}"

    print(f"{SYSTEM_PROMPT}\n{user_message}")
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,   # Low = factual;  High = creative
    )
    return response.choices[0].message.content, context

In [60]:
answer, context = rag("What are the primary topics covered in this document?")
print(answer)
print(f"{250*'='}")
print(f"\n\nSOURCES:\n {context}")

You are a helpful HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information.
Context:
[Source 1]
AtliqAI HR Policies AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining. --- Employment & Onboarding Offer and Joining Formalities Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will share a pre-joining checklist that includes submission of educational certificates, identity proof, address proof, previous employment documents, and a recent

---

[Source 